# test-E — C2: Docling layout-aware, table-atomic chunking

answering: **gpt-5.6-luna**, embedding: **text-embedding-3-small**.

Means over 3 repeats of 150 in-scope and 20 out-of-scope questions. Reads `answers-r*.json` and `judge-r*.json` in this folder; nothing here calls an LLM.

In [1]:
import sys
from pathlib import Path

import pandas as pd

sys.path.insert(0, str(Path.cwd().parent.parent / "code"))  # test/code
import imka_eval as E

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

RUN = "test-E"
rows = E.run_rows(RUN)
in_scope = rows[rows.group != "out_of_scope"]

## Documents

The five documents the run searched (pinned in `documents.json`).

In [2]:
docs = E._read_json(E.run_dir(RUN) / "documents.json")["documents"]
pd.DataFrame([{"document": d["document_name"][:45], "document_id": d["document_id"], "chunks": d.get("chunk_count"), "captioning": (d.get("models") or {}).get("captioning", "-"), "embedding": (d.get("embedding") or {}).get("model", E.DEFAULT_EMBEDDING)} for d in docs])

,document,document_id,chunks,captioning,embedding
0,ABB Manual for Induction Motors and Generator,fdb5c8bb-ba46-40f5-b9a2-e30841e3a2b4,478,-,text-embedding-3-small
1,EMSA Diesel Generator manual-en.pdf,5e59d8ad-dde6-4c8b-9ca9-471960f2a127,285,-,text-embedding-3-small
2,Kohler Generator Manual.pdf,ad188559-78e5-49df-a136-fd909eb8407b,294,-,text-embedding-3-small
3,Nidec Motor Manual.pdf,ecc5861a-047f-4623-bbf0-679bffbfde8f,116,-,text-embedding-3-small
4,Siemens_Manual_Vertical_Induction_Motors_EN.p,23b6d1a9-0fcc-42f1-85f8-ec43c7060425,119,-,text-embedding-3-small


## Results by question group

In [3]:
E.summary(runs=(RUN,), by="group")

n  hit@1  hit@3  hit@5  reciprocal_rank  correct_pdf@5  correct_pdf@1  candidates_hit  faithfulness  answer_relevancy  context_recall  answer_accuracy  refused
run    group                                                                                                                                                                           
test-E tables          30  0.767  0.956  0.989            0.869          0.989          0.956           1.000         0.907             0.878           0.933            0.931      NaN
       cross_document  30  0.767  0.922  0.944            0.846          1.000          0.833           1.000         0.902             0.839           0.933            0.894      NaN
       topical         30  0.756  0.878  0.933            0.823          0.956          0.878           1.000         0.924             0.782           0.843            0.842      NaN
       paraphrase      35  0.667  0.848  0.924            0.760          0.990          0.914           1.000         0.899             0.767           0.876            0.843      NaN
       figures         25  0.787  0.907  0.920            0.841          1.000          0.987           0.933         0.892             0.406           0.387            0.347      NaN
       out_of_scope    20    NaN    NaN    NaN              NaN            NaN            NaN             NaN           NaN               NaN             NaN              NaN      1.0

In [4]:
E.summary(runs=(RUN,), by=None)

,n,hit@1,hit@3,hit@5,reciprocal_rank,correct_pdf@5,correct_pdf@1,candidates_hit,faithfulness,answer_relevancy,context_recall,answer_accuracy,refused
run,,,,,,,,,,,,,
test-E,170,0.744,0.9,0.942,0.825,0.987,0.911,0.989,0.905,0.747,0.811,0.788,1.0


## Stability across the three repeats

In [5]:
in_scope.groupby("repeat")[["hit@5", "context_recall", "answer_accuracy"]].mean().round(3)

,hit@5,context_recall,answer_accuracy
repeat,,,
1,0.953,0.818,0.792
2,0.953,0.813,0.790
3,0.920,0.801,0.782


## Out-of-scope questions

Share of the 20 out-of-scope questions declined, per repeat.

In [6]:
rows[rows.group == "out_of_scope"].groupby("repeat")["refused"].mean().round(3)

repeat
1    1.0
2    1.0
3    1.0
Name: refused, dtype: float64

## Lowest answer accuracy

The ten in-scope questions with the lowest accuracy, averaged over the repeats, with the first repeat's answer.

In [7]:
answers = {r["question"]: r.get("answer") or "" for r in E.load_answers(RUN, 1)["results"]}
worst = (in_scope.groupby(["question", "group", "document_name"])["answer_accuracy"].mean()
         .sort_values().head(10).reset_index())
worst["document_name"] = worst["document_name"].str[:20]
worst["answer (repeat 1)"] = worst["question"].map(lambda q: answers.get(q, "")[:160])
worst

,question,group,document_name,answer_accuracy,answer (repeat 1)
0,According to the Kohler KD27V12 manual's TBN c...,figures,Kohler Generator Man,0.0,## Answer\n\nThe provided context does not sta...
1,"As a thumb rule, how often does ABB say an ins...",topical,ABB Manual for Induc,0.0,## Answer\n\nABB states that insulation resist...
2,"In Fig. 27 of the Kohler KD27V12 manual, what ...",figures,Kohler Generator Man,0.0,The recommended and minimum TBN values at **0 ...
3,"In Fig. 27 of the Kohler KD27V12 manual, appro...",figures,Kohler Generator Man,0.0,The provided context does not contain enough i...
4,"From Figure 42 of the ABB manual, at 18 brushe...",figures,ABB Manual for Induc,0.0,The approximate maximum cleaning interval **ca...
5,"From Figure 42 of the ABB manual, at 6 brushes...",figures,ABB Manual for Induc,0.0,## Answer\n\nThe approximate maximum cleaning ...
6,How do I work out equivalent running hours for...,paraphrase,ABB Manual for Induc,0.0,## Answer\n\nThe supplied context is insuffici...
7,"From Figure 38 of the ABB manual, approximatel...",figures,ABB Manual for Induc,0.0,"From Figure 38, the coefficient $K_t$ at a win..."
8,What is the correct charge generator belt tens...,figures,EMSA Diesel Generato,0.0,## Charge generator belt tension\n\nThe provid...
9,What symptoms suggest a faulty temperature sen...,paraphrase,ABB Manual for Induc,0.0,## Symptoms and likely indication\n\nA **high ...


## Answering cost and time per question

OpenAI models are priced per token; Qwen per second of model working time (llama-server's prompt + generation time) at `E.GPU_PRICE_PER_HOUR`. RunPod queueing, cold starts and job overhead are shown (`runpod_exec_s`, `runpod_delay_s`) but not charged. `wall_s` is the question's end-to-end time as the evaluation saw it.

In [8]:
E.answering_cost((RUN,)).T

run,test-E
answers,510
calls/q,3.0
prompt_tokens/q,2354
completion_tokens/q,401
model_s/q,None
runpod_exec_s/q,None
runpod_delay_s/q,None
wall_s/q (median),7.4
"cost per 1,000 questions ($)",0.95


## Judge spend (Anthropic)

In [9]:
spend = E.judge_spend((RUN,))
spend.groupby("model")[["calls", "input_tokens", "output_tokens", "cost_usd"]].sum()

,calls,input_tokens,output_tokens,cost_usd
model,,,,
claude-haiku-5-5,2760,5682119,708585,0.9225
claude-sonnet-5-5,900,1167627,9000,2.4253
